# 02 — Undersampling + Train/Test Split

Applies NearMiss v3 undersampling to the majority (non-cytotoxic) class:
- `sampling_strategy=0.2` (1:5 cytotoxic:noncytotoxic ratio)
- `n_neighbors_ver3=50`

Then performs stratified 80/20 train/test split.

Expected record counts after undersampling:
- 3T3:    24,042 total (4,007 cytotoxic + 20,035 non-cytotoxic)
- HEK293: 36,846 total (6,141 cytotoxic + 30,705 non-cytotoxic)

**Note:** NearMiss v3 operates in feature space, so fingerprints must be computed first.
This notebook loads pre-computed ECFP4 fingerprints from notebook 03, OR we compute them inline.

In [ ]:
import os
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem, DataStructs
from imblearn.under_sampling import NearMiss
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

os.makedirs('../data/splits', exist_ok=True)

RANDOM_STATE = 42

In [ ]:
def smiles_to_ecfp4(smiles_list, nbits=1024):
    """Convert list of SMILES to ECFP4 numpy array (radius=2, 1024 bits)."""
    fps = []
    valid_idx = []
    for i, smi in enumerate(smiles_list):
        mol = Chem.MolFromSmiles(str(smi))
        if mol is not None:
            fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=nbits)
            arr = np.zeros((nbits,), dtype=np.uint8)
            DataStructs.ConvertToNumpyArray(fp, arr)
            fps.append(arr)
            valid_idx.append(i)
    return np.array(fps), valid_idx

In [ ]:
DATASETS = ['3T3', 'HEK293']

for name in DATASETS:
    print(f'\n=== {name} ===')

    df = pd.read_csv(f'../data/curated/{name}_curated.csv')
    print(f'Loaded {len(df):,} compounds.')

    # Compute ECFP4 fingerprints for NearMiss (operates in feature space)
    print('Computing ECFP4 fingerprints...')
    X_all, valid_idx = smiles_to_ecfp4(df['SMILES'].tolist())
    y_all = df['label'].values[valid_idx]
    smiles_all = df['SMILES'].values[valid_idx]

    print(f'Valid molecules: {len(X_all):,}')
    print(f'Class distribution: cytotoxic={y_all.sum():,}, non-cytotoxic={(y_all==0).sum():,}')

    # NearMiss v3 undersampling — 1:5 ratio
    print('Applying NearMiss v3 undersampling (sampling_strategy=0.2)...')
    nm = NearMiss(version=3, sampling_strategy=0.2, n_neighbors_ver3=50, n_jobs=-1)
    X_res, y_res = nm.fit_resample(X_all, y_all)

    # Recover SMILES for the resampled indices
    resampled_idx = nm.sample_indices_
    smiles_res = smiles_all[resampled_idx]

    print(f'After undersampling: {len(X_res):,} compounds')
    print(f'  cytotoxic={y_res.sum():,}, non-cytotoxic={(y_res==0).sum():,}')

    # 80/20 stratified split
    X_train, X_test, y_train, y_test, smi_train, smi_test = train_test_split(
        X_res, y_res, smiles_res,
        test_size=0.20, train_size=0.80,
        random_state=RANDOM_STATE, stratify=y_res
    )

    print(f'Train: {len(X_train):,} | Test: {len(X_test):,}')

    # Save splits as CSVs with SMILES + label columns
    pd.DataFrame({'SMILES': smi_train, 'label': y_train}).to_csv(
        f'../data/splits/{name}_train.csv', index=False)
    pd.DataFrame({'SMILES': smi_test, 'label': y_test}).to_csv(
        f'../data/splits/{name}_test.csv', index=False)

    # Save fingerprint arrays
    np.save(f'../data/splits/{name}_X_train.npy', X_train)
    np.save(f'../data/splits/{name}_X_test.npy',  X_test)
    np.save(f'../data/splits/{name}_y_train.npy', y_train)
    np.save(f'../data/splits/{name}_y_test.npy',  y_test)

    print(f'Saved splits to ../data/splits/{name}_*.csv and .npy')